# Copilot Phase 3 — AI-Generated Monthly Risk Reports

**Goal:** Read the portfolio analytics produced in Phase 0 & 2, then call the 
Claude API to generate concise, data-grounded monthly risk reports that a 
bank's executive committee can read in under two minutes.

| Cell | Task |
|---|---|
| 1 | Imports & API setup |
| 2 | Load data (KPI, risk profile, priority loans) |
| 3 | Build structured data context string |
| 4 | Define `generate_monthly_insight()` |
| 5 | Generate Month 3 report (peak-risk month) |
| 6 | Generate all months (trend comparison) |
| 7 | Save reports to `ai_insights/` |

## Cell 1 · Imports & API Setup

Install `anthropic` if needed, then initialise the client. 
A quick connectivity check confirms the API key is valid before any data work.

In [11]:
# Install anthropic if not already present
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'anthropic', '-q'],
               check=True)

import anthropic
import pandas as pd
import json
from pathlib import Path

pd.set_option('display.float_format', '{:.4f}'.format)

DATA_DIR    = Path('.')
REPORTS_DIR = DATA_DIR / 'ai_insights'
REPORTS_DIR.mkdir(exist_ok=True)

# ── API client ──────────────────────────────────────────────────────────────
api_key = 'YOUR_API_KEY_HERE'
client  = anthropic.Anthropic(api_key=api_key)

# ── Connectivity check ──────────────────────────────────────────────────────
try:
    ping = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=10,
        messages=[{'role': 'user', 'content': 'ping'}]
    )
    print('✓ API connection OK  |  model:', ping.model)
except anthropic.AuthenticationError:
    print('✗ API key invalid — update api_key above and re-run')
except anthropic.APIConnectionError as e:
    print(f'✗ Network error: {e}')
except Exception as e:
    print(f'✗ Unexpected error: {type(e).__name__}: {e}')

✓ API connection OK  |  model: claude-haiku-4-5-20251001


## Cell 2 · Load Data

Three CSV files produced by Phase 0 and Phase 2. 
Month 6 and 7 are excluded — each has fewer than 250 active loans, 
which makes any rate-based KPI statistically unreliable.

In [12]:
# ── Portfolio-level monthly KPI (Phase 0 output) ──────────────────────────
kpi_raw = pd.read_csv(DATA_DIR / 'portfolio_monthly_kpi.csv')
kpi     = kpi_raw[kpi_raw['month_number'] <= 5].reset_index(drop=True)

# ── Risk band profile (Phase 2 output) ────────────────────────────────────
risk_profile = pd.read_csv(DATA_DIR / 'risk_profile.csv')

# ── Top-50 priority loans (Phase 2 output) ────────────────────────────────
priority_loans = pd.read_csv(DATA_DIR / 'priority_loans.csv')

print('Loaded:')
print(f'  portfolio_monthly_kpi  : {kpi.shape}    (months {kpi.month_number.min()}-{kpi.month_number.max()})')
print(f'  risk_profile           : {risk_profile.shape}')
print(f'  priority_loans         : {priority_loans.shape}')
print()
print('KPI preview:')
print(kpi[['month_number','representative_month','active_loans',
           'avg_on_time_rate','pct_delinquent','pct_high_risk',
           'deteriorating_count','deteriorating_pct']].to_string(index=False))

Loaded:
  portfolio_monthly_kpi  : (5, 10)    (months 1-5)
  risk_profile           : (4, 12)
  priority_loans         : (50, 11)

KPI preview:
 month_number representative_month  active_loans  avg_on_time_rate  pct_delinquent  pct_high_risk  deteriorating_count  deteriorating_pct
            1              2023-01        183596            0.9820          0.0180         0.0180                    0             0.0000
            2              2023-02        183330            0.9518          0.0865         0.0865                12669             0.0691
            3              2023-03        170919            0.9466          0.1211         0.0391                16085             0.0941
            4              2023-04         90423            0.9733          0.0757         0.0310                 4907             0.0543
            5              2023-05         11599            0.9933          0.0239         0.0000                  189             0.0163


## Cell 3 · Build Structured Data Context String

Rather than dumping raw CSV text into the prompt, we build a compact, 
human-readable context block. This keeps the prompt concise and forces 
the model to reason over specific numbers rather than scan a table.

In [13]:
def build_data_context(target_month: int) -> str:
    """Return a structured text context for the target month's risk report."""

    # ── Section 1: Portfolio KPI trend (all 5 months for context) ────────
    kpi_lines = []
    for _, row in kpi.iterrows():
        marker = ' ← TARGET' if row['month_number'] == target_month else ''
        kpi_lines.append(
            f"  Month {int(row['month_number'])} ({row['representative_month']}): "
            f"active_loans={int(row['active_loans']):,}, "
            f"avg_on_time_rate={row['avg_on_time_rate']:.4f}, "
            f"pct_delinquent={row['pct_delinquent']:.4f}, "
            f"pct_high_risk={row['pct_high_risk']:.4f}, "
            f"deteriorating_loans={int(row['deteriorating_count']):,}, "
            f"deteriorating_pct={row['deteriorating_pct']:.4f}"
            f"{marker}"
        )
    kpi_block = 'PORTFOLIO KPI TREND (Months 1-5):\n' + '\n'.join(kpi_lines)

    # ── Section 2: Risk band profile (portfolio-wide snapshot) ────────────
    band_lines = []
    for _, row in risk_profile.iterrows():
        band_lines.append(
            f"  {row['band']:<10}: "
            f"loans={int(row['loan_count']):,} ({row['loan_pct']:.1f}%), "
            f"avg_default_proba={row['avg_default_proba']:.4f}, "
            f"avg_on_time_rate={row['avg_on_time_rate']:.4f}, "
            f"avg_principal=${row['avg_principal']:,.0f}, "
            f"avg_days_delinquent={row['avg_days_delinquent']:.2f}"
        )
    band_block = 'RISK BAND PROFILE (portfolio-wide):\n' + '\n'.join(band_lines)

    # ── Section 3: Top-10 priority loans summary ───────────────────────────
    top10 = priority_loans.head(10)
    p_lines = []
    for _, row in top10.iterrows():
        p_lines.append(
            f"  {row['loan_id']}: "
            f"proba={row['predicted_default_proba']:.4f}, "
            f"principal=${row['principal']:,.0f}, "
            f"on_time_rate={row['on_time_rate_cumul']:.4f}, "
            f"days_delinquent={row['total_delinquent_cumul']:.1f}, "
            f"loan_type={row['loan_type_clean']}, "
            f"state={row['state']}"
        )
    priority_block = 'TOP-10 INTERVENTION PRIORITY LOANS:\n' + '\n'.join(p_lines)

    # ── Combine ────────────────────────────────────────────────────────────
    return '\n\n'.join([kpi_block, band_block, priority_block])


# Preview for Month 3
sample_context = build_data_context(3)
print(sample_context)

PORTFOLIO KPI TREND (Months 1-5):
  Month 1 (2023-01): active_loans=183,596, avg_on_time_rate=0.9820, pct_delinquent=0.0180, pct_high_risk=0.0180, deteriorating_loans=0, deteriorating_pct=0.0000
  Month 2 (2023-02): active_loans=183,330, avg_on_time_rate=0.9518, pct_delinquent=0.0865, pct_high_risk=0.0865, deteriorating_loans=12,669, deteriorating_pct=0.0691
  Month 3 (2023-03): active_loans=170,919, avg_on_time_rate=0.9466, pct_delinquent=0.1211, pct_high_risk=0.0391, deteriorating_loans=16,085, deteriorating_pct=0.0941 ← TARGET
  Month 4 (2023-04): active_loans=90,423, avg_on_time_rate=0.9733, pct_delinquent=0.0757, pct_high_risk=0.0310, deteriorating_loans=4,907, deteriorating_pct=0.0543
  Month 5 (2023-05): active_loans=11,599, avg_on_time_rate=0.9933, pct_delinquent=0.0239, pct_high_risk=0.0000, deteriorating_loans=189, deteriorating_pct=0.0163

RISK BAND PROFILE (portfolio-wide):
  Low       : loans=146,879 (80.0%), avg_default_proba=0.0264, avg_on_time_rate=1.0000, avg_principal

## Cell 4 · Define `generate_monthly_insight()`

A single function that wraps the API call. 
The system prompt establishes the persona; the user message delivers 
the data and the exact output structure we want.
Error handling surfaces the three most likely failure modes clearly.

In [14]:
SYSTEM_PROMPT = """You are a senior risk analyst at a commercial bank. \
You write concise monthly portfolio risk reports for the executive committee. \
Your tone is direct, data-driven, and action-oriented. \
Avoid generic statements — every sentence must reference specific numbers from the data provided."""


def generate_monthly_insight(month_number: int) -> str:
    """
    Generate an executive-facing monthly risk report for the given month.

    Returns the report as a string, or an error message if the API call fails.
    """
    # Pull the target month's KPI row for inline reference
    m_row = kpi[kpi['month_number'] == month_number].iloc[0]
    cal   = m_row['representative_month']

    data_context = build_data_context(month_number)

    user_message = f"""Generate a monthly portfolio risk report for Month {month_number} ({cal}).

DATA CONTEXT:
{data_context}

Produce exactly three sections:

## Executive Summary
(One paragraph, 3-4 sentences. State the overall portfolio health for Month {month_number}, 
call out the single most important risk signal, and compare to the prior month where relevant.)

## Key Risk Signals
(2-3 bullet points. Each must cite a specific number from the data above. 
Focus on what changed or what is anomalous — not what is stable.)

## Recommended Actions
(2-3 concrete, actionable recommendations. Name specific loan segments, 
thresholds, or loan IDs from the priority list where appropriate. 
Each action should be implementable by a collections or risk team within 30 days.)"""

    try:
        response = client.messages.create(
            model      = 'claude-haiku-4-5-20251001',
            max_tokens = 1024,
            system     = SYSTEM_PROMPT,
            messages   = [{'role': 'user', 'content': user_message}]
        )
        report = response.content[0].text

    except anthropic.AuthenticationError:
        report = '[ERROR] API key invalid. Update api_key in Cell 1 and restart kernel.'
    except anthropic.RateLimitError:
        report = '[ERROR] Rate limit hit. Wait 60 seconds and retry.'
    except anthropic.APIConnectionError as e:
        report = f'[ERROR] Network error: {e}'
    except Exception as e:
        report = f'[ERROR] Unexpected: {type(e).__name__}: {e}'

    # ── Print with header ───────────────────────────────────────────────────
    header = f'{'='*70}\nMONTHLY RISK REPORT — Month {month_number} ({cal})\n{'='*70}'
    print(header)
    print(report)
    print()
    return report


print('generate_monthly_insight() defined and ready.')

generate_monthly_insight() defined and ready.


## Cell 5 · Generate Month 3 Report

Month 3 is the highest-risk month in the dataset: 
`pct_delinquent = 12.1%`, `deteriorating_loans = 16,085` (peak). 
It produces the most information-rich report and best demonstrates 
the system's ability to surface actionable insights.

In [15]:
report_m3 = generate_monthly_insight(3)

MONTHLY RISK REPORT — Month 3 (2023-03)
## Executive Summary

Month 3 portfolio health has deteriorated materially, with active loans declining 6.8% to 170,919 and delinquency spiking 40% month-over-month to 12.11%. The critical risk band now comprises 15.0% of the portfolio (27,585 loans) with a weighted average default probability of 0.9981 and on-time rate of only 51.85%, representing the single most important risk signal. While high-risk loans decreased from 8.65% to 3.91%, the surge in deteriorating loans to 16,085 (9.41% of portfolio) indicates accelerating credit quality degradation that offsets any gains in portfolio reduction.

## Key Risk Signals

- **Critical band concentration reached 15.0% threshold:** 27,585 loans now classified critical, averaging 3.59 days delinquent with $74,195 principal per loan and near-certain default probability of 0.9981, up from negligible levels in Month 1.

- **Delinquency rate doubled in two months:** Portfolio delinquency jumped from 1.80% (

## Cell 6 · Generate All Months (Trend Comparison)

Loop through Months 1–5 to produce a full trend narrative. 
Month 1 serves as the baseline; the progression through Months 2–4 
captures the deterioration peak and recovery that makes this dataset 
analytically interesting.

In [16]:
reports = {}   # month_number → report text

for m in range(1, 6):
    print(f'Generating Month {m} report...')
    reports[m] = generate_monthly_insight(m)

print(f'\n✓ Generated {len(reports)} reports.')

Generating Month 1 report...
MONTHLY RISK REPORT — Month 1 (2023-01)
## Executive Summary

Portfolio health in January 2023 is strong across primary metrics: 183,596 active loans show a 98.20% on-time payment rate with only 1.80% delinquency and 1.80% high-risk concentration. However, the Critical risk band represents a material concentration risk, comprising 15.0% of the portfolio (27,585 loans) with a 99.81% average default probability and significantly degraded on-time performance at 51.85%. This bifurcated profile—healthy mainstream portfolio offsetting a distressed tail—requires immediate intervention on the Critical segment before deterioration accelerates across Month 2.

## Key Risk Signals

- **Critical band default concentration**: 27,585 loans (15.0% of portfolio) carry 0.9981 average default probability with on-time rate of only 0.5185, representing ~$2.0 billion in principal at extreme risk ($74,195 average principal × 27,585 loans).

- **Top-10 loans at certainty of defau

## Cell 7 · Save Reports

Write each report to `ai_insights/month_N_risk_report.txt`. 
The directory is created if it doesn't exist.

In [17]:
# If Cell 6 was skipped, make sure report_m3 is at least in reports
if not reports:
    reports[3] = report_m3

saved = []
for month_num, report_text in sorted(reports.items()):
    m_row    = kpi[kpi['month_number'] == month_num].iloc[0]
    cal      = m_row['representative_month']
    filename = REPORTS_DIR / f'month_{month_num}_{cal}_risk_report.txt'

    with open(filename, 'w', encoding='utf-8') as f:
        f.write(f'MONTHLY RISK REPORT — Month {month_num} ({cal})\n')
        f.write('='*70 + '\n\n')
        f.write(report_text)
        f.write('\n')

    saved.append(filename)
    print(f'  ✓ Saved: {filename.name}')

print(f'\nAll reports saved to: {REPORTS_DIR.resolve()}')

  ✓ Saved: month_1_2023-01_risk_report.txt
  ✓ Saved: month_2_2023-02_risk_report.txt
  ✓ Saved: month_3_2023-03_risk_report.txt
  ✓ Saved: month_4_2023-04_risk_report.txt
  ✓ Saved: month_5_2023-05_risk_report.txt

All reports saved to: /Users/apple/Library/CloudStorage/OneDrive-MissouriStateUniversity/AITCC/赛/Analyze/Loan/ai_insights
